# Road Pixels in Link Corridors

Most link corridors contain at least 10–12% settlement, likely the road the link runs on. This notebook checks which `Basemap05` settlement classes appear in a narrow strip around each link vs. the rest of the corridor. It also counts how many links have no mapped road in their strip, and how the road share changes with strip width. The goal is to decide whether to exclude road pixels near the link, and how.

<br><br>

## 0. Setup

### 0.1. Imports

In [1]:
import pyogrio
import numpy as np
import pandas as pd

from splendor.io import load_network, load_raster
from splendor.paths import network_path, basemap05_lu01_path, BASEMAP05_LU01_VAT
from splendor.config import STUDY_AREA, CORRIDOR_RADIUS
from splendor.landuse import get_link_corridors, get_pixel_values, ROAD_CODE

### 0.2. Data

In [2]:
_, edges = load_network(network_path(STUDY_AREA))
raster, transform, _ = load_raster(basemap05_lu01_path(STUDY_AREA))
vat = pyogrio.read_dataframe(BASEMAP05_LU01_VAT)

<br><br>

## 1. Get Settlement Classes

In [3]:
def get_main_type(series: pd.Series) -> pd.Series:
    return series.astype(int) // 1000

# Pixels per raw value in the clip (0 = no data)
values, counts = np.unique(raster, return_counts=True)
pixel_counts = pd.DataFrame({"Value": values, "n_pixels": counts})
pixel_counts = pixel_counts[pixel_counts["Value"] != 0]

# Attach class code and name from the VAT
pixel_counts = pixel_counts.merge(vat[["Value", "C_26", "C_27"]], on="Value")

# Keep settlement (main type 101)
settlement = pixel_counts[get_main_type(pixel_counts["C_26"]) == 101]

# Collapse combination values to classes
settlement_classes = (
    settlement.groupby(["C_26", "C_27"], as_index=False)["n_pixels"]
    .sum()
    .sort_values("n_pixels", ascending=False)
    .reset_index(drop=True)
)
settlement_classes["share"] = settlement_classes["n_pixels"] / settlement_classes["n_pixels"].sum()
settlement_classes


,C_26,C_27,n_pixels,share
0,101301.0,101301 Road and rail networks and associated land,900307,0.446862
1,101201.0,101201 Discontinuous residential area,873992,0.433801
2,101402.0,101402 Sports and recreation sites,79962,0.039689
3,101202.0,101202 Discontinuous commercial and industrial...,65935,0.032726
4,101101.0,101101 Continuous residential area,50852,0.025240
5,101102.0,101102 Continuous commercial and industrial area,24553,0.012187
6,101305.0,101305 Mineral extraction sites (excluding pea...,9894,0.004911
7,101502.0,101502 Cemeteries,6817,0.003384
8,101303.0,101303 Airports,2418,0.001200


<br><br>

## 2. Build Link Zones

In [4]:
STRIP_RADII = [5, 10, 15, 20] # Meters, half-width of the strip around each link

corridors = get_link_corridors(edges, CORRIDOR_RADIUS)

# Per strip radius: the strip itself and the rest of the corridor outside it
strips = {r: get_link_corridors(edges, r) for r in STRIP_RADII}
rests = {r: corridors.difference(strips[r]) for r in STRIP_RADII}

<br><br>

## 3. Sample Raw Values

In [5]:
zone_pixels = []
for r in STRIP_RADII:
    for zone, geometries in [("strip", strips[r]), ("rest", rests[r])]:
        for edge_id, geom in geometries.items():
            values = get_pixel_values(geom, raster, transform)
            # Count pixels per raw value to keep table small
            pixel_values, n_pixels = np.unique(values, return_counts=True)

            for pixel_val, n in zip(pixel_values, n_pixels):
                # Drop no data
                if pixel_val == 0:
                    continue

                zone_pixels.append({
                    "edge_id": edge_id,
                    "strip_r": r,
                    "zone": zone,
                    "Value": pixel_val,
                    "n_pixels": n
                })

zone_pixels = pd.DataFrame(zone_pixels)

# Attach class code and name plus the main type for filtering
zone_pixels = zone_pixels.merge(vat[["Value", "C_26", "C_27"]], on="Value")
zone_pixels["main_type"] = get_main_type(zone_pixels["C_26"])
zone_pixels

# Collapse the "Value" dimension, i.e., group by type
zone_pixels = (
    zone_pixels.groupby(
        ["edge_id", "strip_r", "zone", "C_26", "C_27", "main_type"],
        as_index=False
    )["n_pixels"]
    .sum()
    .reset_index(drop=True)
)

# Add road flag
zone_pixels["road_pixels"] = zone_pixels["n_pixels"] * (zone_pixels["C_26"].astype(int) == ROAD_CODE)
zone_pixels

,edge_id,strip_r,zone,C_26,C_27,main_type,n_pixels,road_pixels
0,0,5,rest,100000.0,100000 Terrestrial (not classified),100,689,0
1,0,5,rest,101101.0,101101 Continuous residential area,101,78,0
2,0,5,rest,101102.0,101102 Continuous commercial and industrial area,101,50,0
3,0,5,rest,101201.0,101201 Discontinuous residential area,101,7253,0
4,0,5,rest,101202.0,101202 Discontinuous commercial and industrial...,101,127,0
...,...,...,...,...,...,...,...,...
44689,637,20,strip,102101.0,102101 Cereals excluding rice (C1000) excludin...,102,110,0
44690,637,20,strip,102104.0,"102104 Root crops, like sugar beet and potatoe...",102,46,0
44691,637,20,strip,102108.0,102108 Fallow land (Q0000),102,8,0
44692,637,20,strip,104000.0,104000 Forest undefined,104,7,0


<br><br>

## 4. Analysis

In [6]:
# Compare road share of settlement pixels in strip vs rest
settlement_pixels = zone_pixels[zone_pixels["main_type"] == 101]

# Combine road shares
totals = settlement_pixels.groupby(["strip_r", "zone"])[["road_pixels", "n_pixels"]].sum()
settlement_road_share = (totals["road_pixels"] / totals["n_pixels"]).unstack().round(4)
settlement_road_share

zone,rest,strip
strip_r,,
5,0.3177,0.9948
10,0.2639,0.9065
15,0.2558,0.8161
20,0.2559,0.7437


In [7]:
# Links with no road in their strip
strip_totals = (
    zone_pixels[zone_pixels["zone"] == "strip"]
    .groupby(["strip_r", "edge_id"])[["road_pixels", "n_pixels"]]
    .sum()
)
link_road_share = strip_totals["road_pixels"] / strip_totals["n_pixels"]
(link_road_share == 0).groupby("strip_r").sum().rename("n_links_without_road")

strip_r
5     0
10    0
15    0
20    0
Name: n_links_without_road, dtype: int64

In [8]:
# How does overall road share in the rest change with the strip radius
overall_totals = zone_pixels.groupby(["strip_r", "zone"])[["road_pixels", "n_pixels"]].sum()
(overall_totals["road_pixels"] / overall_totals["n_pixels"]).unstack().round(4)

zone,rest,strip
strip_r,,
5,0.0693,0.9920
10,0.0536,0.6706
15,0.0506,0.4820
20,0.0496,0.3785


<br><br>

## 5. Conclusion

We see that the settlement in every corridor is mostly the link's own road (class `101301`, "Road and rail networks and associated land"). From a 10 meter strip radius on, the road share is the rest of the corridor stays constant (approx. 5% of all pixels, approx. 26% of settlemtn), which the background of other roads. At 5 meters, it's higher (approx. 7% and 32%), suggesting that the road "leaks" out of the strip. At every radius, every link's strip contains mapped road.

We thus decide to ignore road pixels within 10 meters of the link.